# GeoVision: train the building-damage model

**What this notebook does:** it cuts out building images from the xView2 satellite data, trains an AI model to tell
four damage levels apart (`no-damage`, `minor-damage`, `major-damage`, `destroyed`), tests it on **disasters it never saw**,
and exports a file (`model.onnx`) that the website can use.

## Before you run it (Kaggle settings)
1. Right panel -> **Session options -> Accelerator -> GPU** (T4 or P100).
2. Right panel -> **Session options -> Internet -> On** (needed once to download the pretrained model weights).
3. **Add Input** -> search `xview2 challenge dataset train and test` (by **tunguz**) -> Add.
4. Run the notebook **top to bottom**. Leave `QUICK_TEST = True` the first time. It is a small test run (about 10-15 min) that proves everything works.
5. If the quick test works: set `QUICK_TEST = False` in the first code cell, then **Run All** again for the real training run.

The final cell makes `geovision_result.zip`. Download it from the **Output** tab on the right.

In [ ]:
# ===== 1. SETTINGS (change these, then run the cells from top to bottom) =====
import os, re, sys, json, random, time, shutil, subprocess, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kw):
        return x

if not os.environ.get("GV_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=False)

QUICK_TEST = True        # True = tiny fast test run. Set to False for the real training run.
MODE = "pair"            # "pair" = BEFORE + AFTER images (more accurate), "post" = AFTER image only
BACKBONE = "resnet18"    # "resnet18" = small and fast for the website, "resnet34" = bigger
TEST_EVENTS = ["hurricane-matthew", "santa-rosa-wildfire"]   # disasters kept 100% out of training

IMG = 224
EPOCHS = 2 if QUICK_TEST else 12
BATCH = 64
CAP = {"train": 300, "val": 100, "test": 100} if QUICK_TEST else {"train": 12000, "val": 2000, "test": 3000}
VAL_IMAGE_FRACTION = 0.15   # share of satellite images (in each training disaster) used for validation
MIN_SIDE_PX = 8             # ignore buildings smaller than this
CONTEXT = 1.6               # crop = building size x 1.6 (a little surrounding context)
MIN_CROP_PX = 64            # but never smaller than 64 px
SEED = 42

INPUT_ROOT = Path(os.environ.get("GV_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("GV_WORK", "/kaggle/working"))
CROP_DIR = WORK / "crops"
RESULT_DIR = WORK / "geovision_result"
RUN_DIR = RESULT_DIR / f"{MODE}_{BACKBONE}"
for d in (CROP_DIR, RESULT_DIR, RUN_DIR, RESULT_DIR / "splits"):
    d.mkdir(parents=True, exist_ok=True)

CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]
C2I = {c: i for i, c in enumerate(CLASSES)}
random.seed(SEED)
np.random.seed(SEED)
print("QUICK_TEST =", QUICK_TEST, "| MODE =", MODE, "| BACKBONE =", BACKBONE)

## 2. Find the dataset and read the building labels
Each satellite image has a BEFORE (`pre_disaster`) and AFTER (`post_disaster`) picture. The label file lists every building
with its outline and damage level.

In [ ]:
# ===== 2. FIND THE DATASET AND READ ALL BUILDING LABELS =====
NUM = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")

def bbox_from_wkt(text):
    # turns "POLYGON ((x y, x y, ...))" into the building's bounding box in pixels
    v = [float(t) for t in NUM.findall(text)]
    xs, ys = v[0::2], v[1::2]
    return min(xs), min(ys), max(xs), max(ys)

pairs = []
for root, dirs, files in os.walk(INPUT_ROOT):
    parent = os.path.dirname(root)
    if os.path.basename(root) == "images" and os.path.isdir(os.path.join(parent, "labels")):
        pairs.append((Path(root), Path(parent) / "labels"))
        dirs[:] = []
if not pairs:
    raise FileNotFoundError(
        "No xView2 images/labels folders found. Click 'Add Input' on the right and add the dataset "
        "'xview2-challenge-dataset-train-and-test' by tunguz."
    )

rows, seen, bad_files = [], set(), 0
for img_dir, lab_dir in pairs:
    before = len(rows)
    for lp in sorted(lab_dir.glob("*_post_disaster.json")):
        image_id = lp.name[: -len("_post_disaster.json")]
        if image_id in seen:
            continue
        post_p = img_dir / f"{image_id}_post_disaster.png"
        pre_p = img_dir / f"{image_id}_pre_disaster.png"
        if not (post_p.exists() and pre_p.exists()):
            continue
        try:
            items = json.loads(lp.read_text(encoding="utf-8"))["features"]["xy"]
        except Exception:
            bad_files += 1
            continue
        disaster = re.sub(r"_\d+$", "", image_id)
        found = False
        for b in items:
            props = b.get("properties", {})
            label = props.get("subtype")
            if props.get("feature_type") != "building" or label not in CLASSES or not b.get("wkt"):
                continue
            try:
                x0, y0, x1, y1 = bbox_from_wkt(b["wkt"])
            except Exception:
                continue
            rows.append(dict(disaster=disaster, image_id=image_id, key=f"{image_id}__{props.get('uid')}",
                             label=label, x0=x0, y0=y0, x1=x1, y1=y1,
                             post_path=str(post_p), pre_path=str(pre_p)))
            found = True
        if found:
            seen.add(image_id)
    print(f"{lab_dir}: {len(rows) - before} labelled buildings")

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No labelled buildings were found. Check that the dataset has *_post_disaster.json files.")
print("Total labelled buildings:", len(df), "| unreadable label files:", bad_files)
print("Satellite images:", df.image_id.nunique(), "| disasters:", df.disaster.nunique())
print()
print(pd.crosstab(df.disaster, df.label)[CLASSES].to_string())

## 3. Split the data fairly
* **Test set** = whole disasters the model never sees in training (the honest test).
* **Train / validation** = split by satellite image inside the remaining disasters, so one image is never in both.
* Each class is capped so `no-damage` does not drown out the rare damage classes.

In [ ]:
# ===== 3. BUILD TRAIN / VALIDATION / TEST SPLITS =====
df = df[((df.x1 - df.x0) >= MIN_SIDE_PX) & ((df.y1 - df.y0) >= MIN_SIDE_PX)].reset_index(drop=True)

all_events = sorted(df.disaster.unique())
missing = [e for e in TEST_EVENTS if e not in all_events]
if missing:
    raise ValueError(f"These TEST_EVENTS are not in the data: {missing}. Available disasters: {all_events}")

test_pool = df[df.disaster.isin(TEST_EVENTS)]
rest = df[~df.disaster.isin(TEST_EVENTS)]

rng = np.random.RandomState(SEED)
val_images = set()
for ev, g in rest.groupby("disaster"):
    imgs = np.array(sorted(g.image_id.unique()))
    rng.shuffle(imgs)
    n_val = int(round(len(imgs) * VAL_IMAGE_FRACTION))
    if len(imgs) >= 3:
        n_val = max(n_val, 1)
    val_images.update(imgs[:n_val].tolist())

train_pool = rest[~rest.image_id.isin(val_images)]
val_pool = rest[rest.image_id.isin(val_images)]

def cap_per_class(d, cap, seed):
    parts = [g.sample(n=min(len(g), cap), random_state=seed) for _, g in d.groupby("label")]
    return pd.concat(parts).reset_index(drop=True)

splits = {
    "train": cap_per_class(train_pool, CAP["train"], SEED),
    "val": cap_per_class(val_pool, CAP["val"], SEED),
    "test": cap_per_class(test_pool, CAP["test"], SEED),
}
for name, d in splits.items():
    d["split"] = name

# Safety checks: no leakage
im = {k: set(v.image_id) for k, v in splits.items()}
assert not (im["train"] & im["val"]), "an image is in both train and val!"
assert not (im["train"] & im["test"]) and not (im["val"] & im["test"]), "an image is in test and train/val!"
assert not (set(splits["train"].disaster) | set(splits["val"].disaster)) & set(TEST_EVENTS), "a test disaster leaked into training!"

summary = pd.DataFrame({k: v.label.value_counts() for k, v in splits.items()}).reindex(CLASSES).fillna(0).astype(int)
print(summary.to_string())
print()
print("Test disasters (never seen in training):", TEST_EVENTS)
print("Training disasters:", sorted(set(splits["train"].disaster)))

## 4. Cut out the building images
For every building we cut a square around it from the BEFORE and the AFTER picture and save it as a 224x224 image.
This is slow the first time and is skipped for crops that already exist.

In [ ]:
# ===== 4. CUT OUT BUILDING CROPS (BEFORE and AFTER) =====
def crop_box(x0, y0, x1, y1, W, H):
    side = int(max(MIN_CROP_PX, CONTEXT * max(x1 - x0, y1 - y0)))
    side = min(side, W, H)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    left = int(round(cx - side / 2))
    top = int(round(cy - side / 2))
    left = max(0, min(left, W - side))
    top = max(0, min(top, H - side))
    return (left, top, left + side, top + side)

def extract(split_df, split):
    (CROP_DIR / split / "post").mkdir(parents=True, exist_ok=True)
    (CROP_DIR / split / "pre").mkdir(parents=True, exist_ok=True)
    failed = 0
    for image_id, g in tqdm(split_df.groupby("image_id"), desc=f"cropping {split}"):
        todo = [r for r in g.itertuples()
                if not ((CROP_DIR / split / "post" / f"{r.key}.jpg").exists()
                        and (CROP_DIR / split / "pre" / f"{r.key}.jpg").exists())]
        if not todo:
            continue
        try:
            with Image.open(g.post_path.iloc[0]) as a:
                post_im = a.convert("RGB")
            with Image.open(g.pre_path.iloc[0]) as b:
                pre_im = b.convert("RGB")
            if pre_im.size != post_im.size:
                pre_im = pre_im.resize(post_im.size, Image.BICUBIC)
            W, H = post_im.size
            for r in todo:
                box = crop_box(r.x0, r.y0, r.x1, r.y1, W, H)
                post_im.crop(box).resize((IMG, IMG), Image.BICUBIC).save(CROP_DIR / split / "post" / f"{r.key}.jpg", quality=95)
                pre_im.crop(box).resize((IMG, IMG), Image.BICUBIC).save(CROP_DIR / split / "pre" / f"{r.key}.jpg", quality=95)
        except Exception as e:
            failed += 1
            print("failed:", image_id, e)
    return failed

final = {}
t0 = time.time()
for name, d in splits.items():
    failed = extract(d, name)
    d = d.copy()
    d["post_crop"] = [f"{name}/post/{k}.jpg" for k in d.key]
    d["pre_crop"] = [f"{name}/pre/{k}.jpg" for k in d.key]
    ok = [(CROP_DIR / p).exists() and (CROP_DIR / q).exists() for p, q in zip(d.post_crop, d.pre_crop)]
    d = d[ok].reset_index(drop=True)
    final[name] = d
    print(f"{name}: {len(d)} crops ready ({failed} images failed)")
print(f"Cropping took {(time.time() - t0) / 60:.1f} min")

# Save the exact split lists (commit these to GitHub so the results can be reproduced)
keep = ["split", "disaster", "image_id", "key", "label", "x0", "y0", "x1", "y1", "post_crop", "pre_crop"]
for name, d in final.items():
    d[keep].to_csv(RESULT_DIR / "splits" / f"{name}.csv", index=False)
train_df, val_df, test_df = final["train"], final["val"], final["test"]

In [ ]:
# ===== 4b. LOOK AT A FEW EXAMPLES (BEFORE | AFTER) =====
fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for r, cls in enumerate(CLASSES):
    g = train_df[train_df.label == cls]
    for j in range(2):
        if len(g) == 0:
            continue
        row = g.iloc[(j * 7919) % len(g)]
        axes[r][2 * j].imshow(Image.open(CROP_DIR / row.pre_crop))
        axes[r][2 * j].set_title("before", fontsize=9)
        axes[r][2 * j + 1].imshow(Image.open(CROP_DIR / row.post_crop))
        axes[r][2 * j + 1].set_title(f"after: {cls}", fontsize=9)
for a in axes.ravel():
    a.axis("off")
plt.tight_layout()
plt.savefig(RUN_DIR / "examples.png", dpi=110)
plt.show()

## 5. The model
A ResNet pretrained on ImageNet. In `pair` mode the BEFORE and AFTER pictures go through the **same** network and the
model compares what changed. Image normalisation is built into the model, so the website only has to send pixel values between 0 and 1.

In [ ]:
# ===== 5. DATA LOADER AND MODEL =====
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "(NO GPU - training will be very slow. Turn on a GPU accelerator.)")
CH = 6 if MODE == "pair" else 3

def jitter(img):
    b = 1 + random.uniform(-0.15, 0.15)
    c = 1 + random.uniform(-0.15, 0.15)
    m = img.mean()
    return (((img - m) * c + m) * b).clamp(0, 1)

def augment(x):
    # flips and 90-degree turns: the same for BEFORE and AFTER so they stay aligned
    if random.random() < 0.5:
        x = x.flip(2)
    if random.random() < 0.5:
        x = x.flip(1)
    k = random.randint(0, 3)
    if k:
        x = torch.rot90(x, k, (1, 2))
    # small lighting changes: separate for BEFORE and AFTER (real photos differ in lighting)
    return torch.cat([jitter(x[i:i + 3]) for i in range(0, x.shape[0], 3)], 0)

class CropDataset(Dataset):
    def __init__(self, d, train):
        self.post = d.post_crop.tolist()
        self.pre = d.pre_crop.tolist()
        self.y = d.label.map(C2I).values
        self.train = train

    def __len__(self):
        return len(self.y)

    @staticmethod
    def _load(rel):
        arr = np.array(Image.open(CROP_DIR / rel).convert("RGB"))
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __getitem__(self, i):
        post = self._load(self.post[i])
        x = post if MODE == "post" else torch.cat([self._load(self.pre[i]), post], 0)
        x = x.float() / 255.0
        if self.train:
            x = augment(x)
        return x, int(self.y[i])

NUM_WORKERS = min(4, os.cpu_count() or 1)
def make_loader(d, train):
    return DataLoader(CropDataset(d, train), batch_size=BATCH, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=(dev.type == "cuda"),
                      persistent_workers=(train and NUM_WORKERS > 0))

train_loader, val_loader, test_loader = make_loader(train_df, True), make_loader(val_df, False), make_loader(test_df, False)

class DamageNet(nn.Module):
    def __init__(self, backbone, mode, n_classes=4, pretrained=True):
        super().__init__()
        self.mode = mode
        ctor = getattr(torchvision.models, backbone)
        try:
            net = ctor(weights="DEFAULT" if pretrained else None)
            if pretrained:
                print("Loaded pretrained ImageNet weights.")
        except Exception as e:
            print("WARNING: could not download pretrained weights (is Internet turned on?):", e)
            net = ctor(weights=None)
        self.feat = net.fc.in_features
        net.fc = nn.Identity()
        self.backbone = net
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        if mode == "post":
            self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.feat, n_classes))
        else:
            self.head = nn.Sequential(nn.Linear(3 * self.feat, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, x):
        if self.mode == "post":
            return self.head(self.backbone((x - self.mean) / self.std))
        pair = torch.stack([x[:, :3], x[:, 3:]], 1)            # B, 2, 3, H, W  (index 0 = BEFORE, 1 = AFTER)
        f = self.backbone((pair.flatten(0, 1) - self.mean) / self.std)
        f = f.reshape(-1, 2, self.feat)
        f_pre, f_post = f[:, 0], f[:, 1]
        return self.head(torch.cat([f_pre, f_post, (f_post - f_pre).abs()], 1))

model = DamageNet(BACKBONE, MODE).to(dev)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 6. Train
The best epoch (by **validation macro-F1**, which treats all four classes equally) is saved as `best.pt`.
The test disasters are not touched here.

In [ ]:
# ===== 6. TRAINING =====
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

counts = train_df.label.map(C2I).value_counts().reindex(range(4)).fillna(1).values.astype(float)
w = (counts.max() / counts) ** 0.5
w = torch.tensor(w / w.mean(), dtype=torch.float32, device=dev)
criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

head_params = list(model.head.parameters())
opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": 3e-4},
                         {"params": head_params, "lr": 1e-3}], weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-4, 1e-3], total_steps=EPOCHS * len(train_loader), pct_start=0.2)
use_amp = dev.type == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except AttributeError:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def predict(loader):
    model.eval()
    ys, ps, pr = [], [], []
    for xb, yb in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(xb.to(dev, non_blocking=True))
        prob = torch.softmax(logits.float(), 1).cpu()
        ys += yb.tolist()
        ps += prob.argmax(1).tolist()
        pr.append(prob)
    return np.array(ys), np.array(ps), torch.cat(pr).numpy()

history, best_f1 = [], -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, total, n = time.time(), 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item() * len(yb)
        n += len(yb)
    yv, pv, _ = predict(val_loader)
    f1 = f1_score(yv, pv, labels=[0, 1, 2, 3], average="macro", zero_division=0)
    acc = accuracy_score(yv, pv)
    history.append(dict(epoch=epoch, train_loss=total / n, val_acc=acc, val_macro_f1=f1))
    mark = ""
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), RUN_DIR / "best.pt")
        mark = "  <- best so far, saved"
    print(f"epoch {epoch:2d}/{EPOCHS} | loss {total / n:.4f} | val acc {acc:.3f} | val macro-F1 {f1:.3f} | {time.time() - t0:.0f}s{mark}")

pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)
print("Best validation macro-F1:", round(best_f1, 4))

## 7. Final test on disasters the model never saw
Run this **once**. Do not change settings after looking at the test result just to improve it.

In [ ]:
# ===== 7. FINAL TEST (held-out disasters) =====
model.load_state_dict(torch.load(RUN_DIR / "best.pt", map_location=dev))
yt, pt, prob_t = predict(test_loader)

acc = accuracy_score(yt, pt)
macro_f1 = f1_score(yt, pt, labels=[0, 1, 2, 3], average="macro", zero_division=0)
report = classification_report(yt, pt, labels=[0, 1, 2, 3], target_names=CLASSES, output_dict=True, zero_division=0)
print(classification_report(yt, pt, labels=[0, 1, 2, 3], target_names=CLASSES, zero_division=0))
cm = confusion_matrix(yt, pt, labels=range(4))

damaged = yt >= 2
severe_miss = float(((pt == 0) & damaged).sum() / max(damaged.sum(), 1))
within_one = float((np.abs(yt - pt) <= 1).mean())
per_event = {ev: float((pt[(test_df.disaster == ev).values] == yt[(test_df.disaster == ev).values]).mean())
             for ev in TEST_EVENTS if (test_df.disaster == ev).any()}
print(f"Test accuracy: {acc:.4f} | macro-F1: {macro_f1:.4f}")
print(f"Major/destroyed buildings predicted as 'no-damage': {severe_miss:.1%}")
print(f"Prediction within one damage level: {within_one:.1%}")
print("Accuracy per held-out disaster:", per_event)

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.imshow(cm, cmap="Blues")
rowsum = cm.sum(1, keepdims=True).clip(min=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm[i, j]}\n{cm[i, j] / rowsum[i, 0]:.0%}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=9)
ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Test confusion matrix ({MODE}, {BACKBONE})")
plt.tight_layout()
plt.savefig(RUN_DIR / "confusion_matrix_test.png", dpi=130)
plt.show()

metrics = dict(
    mode=MODE, backbone=BACKBONE, quick_test=QUICK_TEST, epochs=EPOCHS, seed=SEED, img_size=IMG,
    test_events=TEST_EVENTS, train_events=sorted(set(train_df.disaster)),
    n_train=len(train_df), n_val=len(val_df), n_test=len(test_df),
    best_val_macro_f1=float(best_f1), test_accuracy=float(acc), test_macro_f1=float(macro_f1),
    major_or_destroyed_predicted_no_damage=severe_miss, within_one_level=within_one,
    per_event_accuracy=per_event, per_class=report, confusion_matrix=cm.tolist(), classes=CLASSES,
    torch=torch.__version__,
)
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

## 8. Export for the website (ONNX)
The website runs this file directly in the browser, so no server is needed. The cell also checks that the exported file
gives the same answers as the PyTorch model.

In [ ]:
# ===== 8. EXPORT TO ONNX AND VERIFY =====
import onnxruntime as ort

cpu_model = DamageNet(BACKBONE, MODE, pretrained=False)
cpu_model.load_state_dict(torch.load(RUN_DIR / "best.pt", map_location="cpu"))
cpu_model.eval()

onnx_path = RUN_DIR / "model.onnx"
dummy = torch.rand(1, CH, IMG, IMG)
kwargs = dict(input_names=["image"], output_names=["logits"],
              dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(cpu_model, dummy, str(onnx_path), dynamo=False, **kwargs)
except TypeError:
    torch.onnx.export(cpu_model, dummy, str(onnx_path), **kwargs)

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
sample = torch.stack([CropDataset(test_df.iloc[:16], False)[i][0] for i in range(min(16, len(test_df)))])
with torch.no_grad():
    ref = cpu_model(sample).numpy()
out = sess.run(None, {"image": sample.numpy()})[0]
max_diff = float(np.abs(ref - out).max())
same_pred = float((ref.argmax(1) == out.argmax(1)).mean())
print(f"ONNX file: {onnx_path.stat().st_size / 1e6:.1f} MB | max logit difference vs PyTorch: {max_diff:.2e} | same predictions: {same_pred:.0%}")
assert max_diff < 1e-2 and same_pred == 1.0, "ONNX output does not match PyTorch - do not use this export"

card = dict(
    classes=CLASSES, mode=MODE, backbone=BACKBONE, img_size=IMG, input_name="image", output_name="logits",
    input_layout=f"[batch, {CH}, {IMG}, {IMG}] float32 RGB values from 0 to 1" +
                 ("; channels 0-2 = BEFORE (pre-disaster) image, channels 3-5 = AFTER (post-disaster) image" if MODE == "pair" else ""),
    normalisation="ImageNet mean/std is applied INSIDE the model - do not normalise in the browser",
    output="raw logits; apply softmax to get probabilities",
    crop_rule=f"square crop centred on the building, side = max({MIN_CROP_PX}, {CONTEXT} x building size) pixels, resized to {IMG}x{IMG}",
    test_macro_f1=float(macro_f1), test_accuracy=float(acc), held_out_disasters=TEST_EVENTS,
)
(RUN_DIR / "model_card.json").write_text(json.dumps(card, indent=2))

## 9. Package the results
Download `geovision_result.zip` from the **Output** tab. It does not include the building crops, so it stays small.

In [ ]:
# ===== 9. PACKAGE THE RESULTS =====
zip_path = WORK / "geovision_result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULT_DIR.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(WORK))
print("Created:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
for p in sorted(RESULT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RESULT_DIR)}  {p.stat().st_size / 1e6:.2f} MB")
if QUICK_TEST:
    print("\nThis was only a QUICK TEST. Set QUICK_TEST = False in the first code cell and run everything again for the real model.")